# **Actividad 3: Introducción a la IA Agentiva**

En las actividades anteriores usamos los Transformers para **generar texto**. En esta actividad damos un paso más: construiremos un **agente** muy simple que no solo genera texto, sino que **decide qué hacer** y **usa herramientas** para responder.

## ¿Qué es un agente?

Un **agente de IA** es un sistema que:

1. **Razona** sobre una tarea (*"¿qué necesito para responder esto?"*).
2. **Actúa** usando herramientas externas (calculadora, búsqueda, API...).
3. **Observa** el resultado de esa acción.
4. **Repite** el ciclo hasta tener la respuesta final.

Este patrón se llama **ReAct** (*Reason + Act*) y es la base de agentes modernos como los que ves en Claude, ChatGPT con tools, LangChain, etc.

## Objetivo de esta actividad

Implementar el patrón ReAct **desde cero**, sin librerías externas, para entender **qué ocurre realmente dentro de un agente**. No es magia: es un bucle.

## **1. El ciclo ReAct**

```
     ┌─────────────────────────────────────────────┐
     │                                             │
     ▼                                             │
  ┌─────────┐    ┌─────────┐    ┌──────────────┐  │
  │ PENSAR  │───▶│ ACTUAR  │───▶│  OBSERVAR    │──┘
  │(Thought)│    │(Action) │    │(Observation) │
  └─────────┘    └─────────┘    └──────────────┘
     │
     ▼
  ┌─────────────┐
  │  RESPUESTA  │
  │   FINAL     │
  └─────────────┘
```

**Ejemplo intuitivo** (cómo respondería una persona):

> **Pregunta:** *"¿Cuánto es el 15% de la población de España?"*
>
> - 🧠 **Pensar:** "Necesito saber cuánta gente vive en España."
> - 🔧 **Actuar:** consultar_poblacion("España")
> - 👁️ **Observar:** 48.000.000
> - 🧠 **Pensar:** "Ahora calculo el 15%."
> - 🔧 **Actuar:** calcular("48000000 * 0.15")
> - 👁️ **Observar:** 7.200.000
> - ✅ **Respuesta:** "El 15% de la población de España son unos 7,2 millones."

Eso es **exactamente** lo que vamos a programar.

## **2. Configuración**

No necesitamos nada nuevo: usaremos únicamente Python estándar. Esto es a propósito — queremos ver el agente "por dentro" sin que una librería (LangChain, AutoGen...) nos lo esconda.

In [1]:
import re       # Para parsear las respuestas del agente
import json     # Para formatear observaciones
import math     # Para la herramienta calculadora

print("✅ Entorno listo. Solo Python estándar.")

✅ Entorno listo. Solo Python estándar.


## **3. Definir las herramientas (tools)**

Una **herramienta** es simplemente una función de Python que el agente puede invocar. Cada herramienta tiene:

- un **nombre** (así la llama el agente),
- una **descripción** (así el agente sabe cuándo usarla),
- una **función** que hace el trabajo real.

Vamos a crear dos herramientas muy sencillas:

| Herramienta | Qué hace |
|---|---|
| `calculadora` | Evalúa una expresión matemática (`"2 + 3 * 4"`). |
| `base_conocimiento` | Devuelve datos de una mini "base de datos" en memoria. |

In [2]:
# --- Herramienta 1: Calculadora ---
def calculadora(expresion: str) -> str:
    """Evalúa una expresión matemática de forma segura."""
    try:
        # Solo permitimos caracteres matemáticos básicos (seguridad)
        if not re.match(r'^[\d\s+\-*/().%]+$', expresion):
            return "Error: expresión no válida."
        resultado = eval(expresion, {"__builtins__": {}}, {})
        return str(resultado)
    except Exception as e:
        return f"Error: {e}"


# --- Herramienta 2: Base de conocimiento simulada ---
# En un agente real esto sería una búsqueda web, una BBDD, una API...
# Aquí es un diccionario para que sea transparente.
BASE_DATOS = {
    "españa":     {"poblacion": 48_000_000, "capital": "Madrid",   "superficie_km2": 505_990},
    "francia":    {"poblacion": 68_000_000, "capital": "París",    "superficie_km2": 643_801},
    "portugal":   {"poblacion": 10_300_000, "capital": "Lisboa",   "superficie_km2":  92_212},
    "italia":     {"poblacion": 59_000_000, "capital": "Roma",     "superficie_km2": 301_340},
    "alemania":   {"poblacion": 84_000_000, "capital": "Berlín",   "superficie_km2": 357_588},
}

def base_conocimiento(consulta: str) -> str:
    """Busca información sobre un país. Formato: 'pais:campo' (ej: 'españa:poblacion')."""
    try:
        pais, campo = consulta.lower().split(":")
        pais, campo = pais.strip(), campo.strip()
        if pais not in BASE_DATOS:
            return f"País '{pais}' no encontrado. Disponibles: {list(BASE_DATOS.keys())}"
        if campo not in BASE_DATOS[pais]:
            return f"Campo '{campo}' no encontrado. Disponibles: {list(BASE_DATOS[pais].keys())}"
        return str(BASE_DATOS[pais][campo])
    except ValueError:
        return "Error: usa el formato 'pais:campo' (ej: 'españa:poblacion')."


# --- Registro de herramientas disponibles para el agente ---
HERRAMIENTAS = {
    "calculadora": {
        "fn": calculadora,
        "descripcion": "Evalúa una expresión matemática. Input: una expresión como '48000000 * 0.15'."
    },
    "base_conocimiento": {
        "fn": base_conocimiento,
        "descripcion": "Consulta datos de un país. Input formato 'pais:campo'. "
                       "Campos disponibles: poblacion, capital, superficie_km2. "
                       "Países: españa, francia, portugal, italia, alemania."
    },
}

# Probemos las herramientas directamente (sin agente todavía)
print("Calculadora:    ", calculadora("48000000 * 0.15"))
print("Base de datos:  ", base_conocimiento("españa:poblacion"))
print("Error esperado: ", base_conocimiento("marte:poblacion"))

Calculadora:     7200000.0
Base de datos:   48000000
Error esperado:  País 'marte' no encontrado. Disponibles: ['españa', 'francia', 'portugal', 'italia', 'alemania']


## **4. El "cerebro" del agente**

En un agente real, el cerebro es un **LLM** (GPT, Claude, Llama...) que lee el historial y decide el siguiente paso.

Aquí vamos a hacer algo **didáctico**: implementaremos un "cerebro" por reglas muy simples que imita cómo decide un LLM. Así puedes ver **el patrón ReAct funcionando** sin depender de un modelo grande y sin esperar a que descargue GB de pesos.

> 💡 **Nota importante:** en la sección 7 (opcional) sustituiremos este cerebro de reglas por un LLM real (`gpt2-spanish`) para que veas la diferencia.

In [3]:
def cerebro_por_reglas(pregunta: str, historial: list) -> str:
    """
    Decide el siguiente paso según el historial.
    Devuelve una cadena con formato ReAct:
      - 'Thought: ... \nAction: tool_name[input]'    (para actuar)
      - 'Thought: ... \nFinal Answer: ...'          (para terminar)
    """
    pregunta_low = pregunta.lower()
    observaciones = [h for h in historial if h.startswith("Observation:")]
    n_obs = len(observaciones)

    # --- CASO 1: pregunta sobre población + porcentaje ---
    # Patrón: "¿Cuánto es el X% de la población de PAIS?"
    m = re.search(r"(\d+)\s*%.*poblaci[oó]n.*de\s+([a-záéíóúñ]+)", pregunta_low)
    if m:
        porcentaje, pais = m.group(1), m.group(2)

        if n_obs == 0:
            # Paso 1: buscar población
            return (f"Thought: Necesito la población de {pais} para calcular el {porcentaje}%.\n"
                    f"Action: base_conocimiento[{pais}:poblacion]")
        elif n_obs == 1:
            # Paso 2: calcular porcentaje con el valor observado
            poblacion = observaciones[0].replace("Observation:", "").strip()
            # Si la observación fue un error, cortamos aquí
            if poblacion.lower().startswith("error") or "no encontrado" in poblacion.lower():
                return (f"Thought: La consulta anterior falló, no puedo continuar.\n"
                        f"Final Answer: No puedo responder: {poblacion}")
            return (f"Thought: Ahora calculo el {porcentaje}% de {poblacion}.\n"
                    f"Action: calculadora[{poblacion} * {porcentaje} / 100]")
        else:
            # Paso 3: respuesta final
            resultado = observaciones[-1].replace("Observation:", "").strip()
            return (f"Thought: Ya tengo el resultado.\n"
                    f"Final Answer: El {porcentaje}% de la población de {pais} es "
                    f"aproximadamente {float(resultado):,.0f} personas.")

    # --- CASO 2: pregunta directa sobre un país ---
    m = re.search(r"(capital|poblaci[oó]n|superficie)\s+de\s+([a-záéíóúñ]+)", pregunta_low)
    if m:
        campo_raw, pais = m.group(1), m.group(2)
        campo = {"capital": "capital", "población": "poblacion",
                 "poblacion": "poblacion", "superficie": "superficie_km2"}[campo_raw]
        if n_obs == 0:
            return (f"Thought: Consulto directamente la base de conocimiento.\n"
                    f"Action: base_conocimiento[{pais}:{campo}]")
        else:
            resultado = observaciones[-1].replace("Observation:", "").strip()
            return f"Thought: Tengo la respuesta.\nFinal Answer: {resultado}"

    # --- CASO 3: cálculo puro ---
    m = re.search(r"cu[aá]nto es\s+(.+?)\??$", pregunta_low)
    if m and n_obs == 0:
        expr = m.group(1).replace("x", "*").replace("por", "*").replace("mas", "+")
        expr = re.sub(r"[^\d+\-*/.() ]", "", expr)
        return (f"Thought: Es un cálculo puro, uso la calculadora.\n"
                f"Action: calculadora[{expr.strip()}]")
    if m and n_obs >= 1:
        return f"Thought: Ya calculé.\nFinal Answer: {observaciones[-1].replace('Observation:', '').strip()}"

    # --- Fallback ---
    return ("Thought: No sé resolver esta pregunta con las herramientas disponibles.\n"
            "Final Answer: No puedo responder esa pregunta con las herramientas que tengo.")

    # --- CASO 4: Conversión de moneda ---
    m = re.search(r"cu[aá]nto son\s+(\d+)\s+euros.*d[oó]lares", pregunta_low)
    if m:
        cantidad = m.group(1)
        if n_obs == 0:
            return (f"Thought: Me piden convertir {cantidad} euros a dólares.\n"
                    f"Action: conversor_moneda[{cantidad}]")
        else:
            resultado = observaciones[-1].replace("Observation:", "").strip()
            return f"Thought: Ya tengo la conversión.\nFinal Answer: {cantidad} euros son {resultado} dólares."

print("✅ Cerebro por reglas definido.")

✅ Cerebro por reglas definido.


## **5. El bucle del agente**

Aquí está **el corazón del agente**. Este bucle es idéntico —conceptualmente— al que usan los agentes más sofisticados del mundo. Lo único que cambia es que el "cerebro" es un LLM en vez de reglas.

```
mientras no termine:
    paso = cerebro(pregunta, historial)       # 🧠 pensar + decidir acción
    si paso contiene "Final Answer":
        devolver respuesta                     # ✅ terminar
    sino:
        resultado = ejecutar_herramienta(paso) # 🔧 actuar
        historial.append(resultado)            # 👁️ observar
```

In [4]:
def ejecutar_accion(accion_str: str) -> str:
    """Parsea 'tool_name[input]' y ejecuta la herramienta."""
    m = re.match(r"(\w+)\[(.*)\]", accion_str.strip())
    if not m:
        return f"Error: no entiendo la acción '{accion_str}'."

    nombre, argumento = m.group(1), m.group(2)
    if nombre not in HERRAMIENTAS:
        return f"Error: herramienta '{nombre}' no existe. Disponibles: {list(HERRAMIENTAS.keys())}"

    return HERRAMIENTAS[nombre]["fn"](argumento)


def agente(pregunta: str, cerebro_fn=cerebro_por_reglas, max_pasos: int = 5, verbose: bool = True):
    """Ejecuta el bucle ReAct hasta obtener una respuesta final."""
    historial = []

    if verbose:
        print(f"\n{'='*60}\n🤔 PREGUNTA: {pregunta}\n{'='*60}")

    for paso in range(1, max_pasos + 1):
        if verbose:
            print(f"\n--- Paso {paso} ---")

        # 🧠 El cerebro decide qué hacer
        decision = cerebro_fn(pregunta, historial)
        if verbose:
            print(decision)

        # ✅ ¿Es una respuesta final?
        if "Final Answer:" in decision:
            respuesta = decision.split("Final Answer:")[-1].strip()
            if verbose:
                print(f"\n🎯 RESPUESTA: {respuesta}")
            return respuesta

        # 🔧 Si no, ejecutamos la acción
        m = re.search(r"Action:\s*(.+)", decision)
        if not m:
            if verbose:
                print("⚠️ No hay acción válida. Abortando.")
            return None

        observacion = ejecutar_accion(m.group(1))
        historial.append(f"Observation: {observacion}")
        if verbose:
            print(f"Observation: {observacion}")

    if verbose:
        print("⚠️ Se alcanzó el número máximo de pasos.")
    return None

print("✅ Bucle agéntico listo.")

✅ Bucle agéntico listo.


## **6. Probar el agente**

Vamos a lanzar el agente con tres preguntas de dificultad creciente:

1. **Pregunta directa** → 1 herramienta.
2. **Pregunta compuesta** → 2 herramientas encadenadas (base de datos → calculadora).
3. **Cálculo puro** → solo calculadora.

In [5]:
# Prueba 1: directa
agente("¿Cuál es la capital de Francia?")


🤔 PREGUNTA: ¿Cuál es la capital de Francia?

--- Paso 1 ---
Thought: Consulto directamente la base de conocimiento.
Action: base_conocimiento[francia:capital]
Observation: París

--- Paso 2 ---
Thought: Tengo la respuesta.
Final Answer: París

🎯 RESPUESTA: París


'París'

In [6]:
# Prueba 2: compuesta — el agente necesita encadenar 2 herramientas
agente("¿Cuánto es el 15% de la población de España?")


🤔 PREGUNTA: ¿Cuánto es el 15% de la población de España?

--- Paso 1 ---
Thought: Necesito la población de españa para calcular el 15%.
Action: base_conocimiento[españa:poblacion]
Observation: 48000000

--- Paso 2 ---
Thought: Ahora calculo el 15% de 48000000.
Action: calculadora[48000000 * 15 / 100]
Observation: 7200000.0

--- Paso 3 ---
Thought: Ya tengo el resultado.
Final Answer: El 15% de la población de españa es aproximadamente 7,200,000 personas.

🎯 RESPUESTA: El 15% de la población de españa es aproximadamente 7,200,000 personas.


'El 15% de la población de españa es aproximadamente 7,200,000 personas.'

In [7]:
# Prueba 3: cálculo puro
agente("¿Cuánto es 1234 * 567?")


🤔 PREGUNTA: ¿Cuánto es 1234 * 567?

--- Paso 1 ---
Thought: Es un cálculo puro, uso la calculadora.
Action: calculadora[1234 * 567]
Observation: 699678

--- Paso 2 ---
Thought: Ya calculé.
Final Answer: 699678

🎯 RESPUESTA: 699678


'699678'

In [8]:
# Prueba 4: pregunta que NO puede responder (sin alucinar)
agente("¿Cuál es la población de Marte?")


🤔 PREGUNTA: ¿Cuál es la población de Marte?

--- Paso 1 ---
Thought: Consulto directamente la base de conocimiento.
Action: base_conocimiento[marte:poblacion]
Observation: País 'marte' no encontrado. Disponibles: ['españa', 'francia', 'portugal', 'italia', 'alemania']

--- Paso 2 ---
Thought: Tengo la respuesta.
Final Answer: País 'marte' no encontrado. Disponibles: ['españa', 'francia', 'portugal', 'italia', 'alemania']

🎯 RESPUESTA: País 'marte' no encontrado. Disponibles: ['españa', 'francia', 'portugal', 'italia', 'alemania']


"País 'marte' no encontrado. Disponibles: ['españa', 'francia', 'portugal', 'italia', 'alemania']"

## **7. Ejercicios**

Ahora te toca a ti. Completa los siguientes ejercicios **modificando el código de arriba**.



### 📝 Ejercicio 1 — Añadir una herramienta nueva

Crea una herramienta `conversor_moneda` que convierta euros a dólares (asume 1€ = 1.08$). Regístrala en `HERRAMIENTAS` y pruébala con una pregunta como *"¿Cuánto son 500 euros en dólares?"*.

Tendrás que ampliar `cerebro_por_reglas` para que reconozca ese patrón.

In [9]:
# --- Nueva herramienta: Conversor de Moneda ---
def conversor_moneda(euros_str: str) -> str:
    """Convierte euros a dólares usando una tasa fija (1€ = 1.08$)."""
    try:
        euros = float(euros_str)
        dolares = euros * 1.08
        return f"{dolares:.2f}"
    except ValueError:
        return "Error: por favor introduce un número válido."

HERRAMIENTAS["conversor_moneda"] = {
    "fn": conversor_moneda,
    "descripcion": "Convierte euros a dólares. Input: cantidad numérica en euros (ej: '500')."
}

In [10]:
def cerebro_por_reglas(pregunta: str, historial: list) -> str:
    """
    Decide el siguiente paso según el historial.
    Devuelve una cadena con formato ReAct:
      - 'Thought: ... \nAction: tool_name[input]'    (para actuar)
      - 'Thought: ... \nFinal Answer: ...'          (para terminar)
    """
    pregunta_low = pregunta.lower()
    observaciones = [h for h in historial if h.startswith("Observation:")]
    n_obs = len(observaciones)

    # --- CASO 1: pregunta sobre población + porcentaje ---
    # Patrón: "¿Cuánto es el X% de la población de PAIS?"
    m = re.search(r"(\d+)\s*%.*poblaci[oó]n.*de\s+([a-záéíóúñ]+)", pregunta_low)
    if m:
        porcentaje, pais = m.group(1), m.group(2)

        if n_obs == 0:
            # Paso 1: buscar población
            return (f"Thought: Necesito la población de {pais} para calcular el {porcentaje}%.\n"
                    f"Action: base_conocimiento[{pais}:poblacion]")
        elif n_obs == 1:
            # Paso 2: calcular porcentaje con el valor observado
            poblacion = observaciones[0].replace("Observation:", "").strip()
            # Si la observación fue un error, cortamos aquí
            if poblacion.lower().startswith("error") or "no encontrado" in poblacion.lower():
                return (f"Thought: La consulta anterior falló, no puedo continuar.\n"
                        f"Final Answer: No puedo responder: {poblacion}")
            return (f"Thought: Ahora calculo el {porcentaje}% de {poblacion}.\n"
                    f"Action: calculadora[{poblacion} * {porcentaje} / 100]")
        else:
            # Paso 3: respuesta final
            resultado = observaciones[-1].replace("Observation:", "").strip()
            return (f"Thought: Ya tengo el resultado.\n"
                    f"Final Answer: El {porcentaje}% de la población de {pais} es "
                    f"aproximadamente {float(resultado):,.0f} personas.")

    # --- CASO 2: pregunta directa sobre un país ---
    m = re.search(r"(capital|poblaci[oó]n|superficie)\s+de\s+([a-záéíóúñ]+)", pregunta_low)
    if m:
        campo_raw, pais = m.group(1), m.group(2)
        campo = {"capital": "capital", "población": "poblacion",
                 "poblacion": "poblacion", "superficie": "superficie_km2"}[campo_raw]
        if n_obs == 0:
            return (f"Thought: Consulto directamente la base de conocimiento.\n"
                    f"Action: base_conocimiento[{pais}:{campo}]")
        else:
            resultado = observaciones[-1].replace("Observation:", "").strip()
            return f"Thought: Tengo la respuesta.\nFinal Answer: {resultado}"

    # --- CASO 3: cálculo puro ---
    m = re.search(r"cu[aá]nto es\s+(.+?)\??$", pregunta_low)
    if m and n_obs == 0:
        expr = m.group(1).replace("x", "*").replace("por", "*").replace("mas", "+")
        expr = re.sub(r"[^\d+\-*/.() ]", "", expr)
        return (f"Thought: Es un cálculo puro, uso la calculadora.\n"
                f"Action: calculadora[{expr.strip()}]")
    if m and n_obs >= 1:
        return f"Thought: Ya calculé.\nFinal Answer: {observaciones[-1].replace('Observation:', '').strip()}"

    # --- CASO 4: Conversión de moneda ---
    m = re.search(r"cu[aá]nto son\s+(\d+)\s+euros.*d[oó]lares", pregunta_low)
    if m:
        cantidad = m.group(1)
        if n_obs == 0:
            return (f"Thought: Me piden convertir {cantidad} euros a dólares.\n"
                    f"Action: conversor_moneda[{cantidad}]")
        else:
            resultado = observaciones[-1].replace("Observation:", "").strip()
            return f"Thought: Ya tengo la conversión.\nFinal Answer: {cantidad} euros son {resultado} dólares."

    # --- Fallback ---
    return ("Thought: No sé resolver esta pregunta con las herramientas disponibles.\n"
            "Final Answer: No puedo responder esa pregunta con las herramientas que tengo.")

print("✅ Cerebro por reglas definido.")

✅ Cerebro por reglas definido.


### 📝 Ejercicio 2 — Ampliar la base de conocimiento

Añade 3 países más a `BASE_DATOS` (por ejemplo: Reino Unido, Japón, Brasil) y prueba preguntas como *"¿Cuánto es el 25% de la población de Japón?"*.



In [11]:
BASE_DATOS.update({
    "reino unido": {"poblacion": 67_000_000, "capital": "Londres", "superficie_km2": 243_610},
    "japón":       {"poblacion": 125_000_000, "capital": "Tokio", "superficie_km2": 377_975},
    "brasil":      {"poblacion": 214_000_000, "capital": "Brasilia", "superficie_km2": 8_515_767}
})

### 📝 Ejercicio 3 — Pregunta compuesta nueva

Diseña una pregunta que requiera **3 herramientas encadenadas** y amplía el cerebro para resolverla. Por ejemplo: *"¿Cuál es la densidad de población de Alemania?"* → necesita población, superficie y calculadora.



In [12]:
def cerebro_por_reglas(pregunta: str, historial: list) -> str:
    """
    Decide el siguiente paso según el historial.
    Devuelve una cadena con formato ReAct:
      - 'Thought: ... \nAction: tool_name[input]'    (para actuar)
      - 'Thought: ... \nFinal Answer: ...'          (para terminar)
    """
    pregunta_low = pregunta.lower()
    observaciones = [h for h in historial if h.startswith("Observation:")]
    n_obs = len(observaciones)

    # --- CASO 1: pregunta sobre población + porcentaje ---
    # Patrón: "¿Cuánto es el X% de la población de PAIS?"
    m = re.search(r"(\d+)\s*%.*poblaci[oó]n.*de\s+([a-záéíóúñ]+)", pregunta_low)
    if m:
        porcentaje, pais = m.group(1), m.group(2)

        if n_obs == 0:
            # Paso 1: buscar población
            return (f"Thought: Necesito la población de {pais} para calcular el {porcentaje}%.\n"
                    f"Action: base_conocimiento[{pais}:poblacion]")
        elif n_obs == 1:
            # Paso 2: calcular porcentaje con el valor observado
            poblacion = observaciones[0].replace("Observation:", "").strip()
            # Si la observación fue un error, cortamos aquí
            if poblacion.lower().startswith("error") or "no encontrado" in poblacion.lower():
                return (f"Thought: La consulta anterior falló, no puedo continuar.\n"
                        f"Final Answer: No puedo responder: {poblacion}")
            return (f"Thought: Ahora calculo el {porcentaje}% de {poblacion}.\n"
                    f"Action: calculadora[{poblacion} * {porcentaje} / 100]")
        else:
            # Paso 3: respuesta final
            resultado = observaciones[-1].replace("Observation:", "").strip()
            return (f"Thought: Ya tengo el resultado.\n"
                    f"Final Answer: El {porcentaje}% de la población de {pais} es "
                    f"aproximadamente {float(resultado):,.0f} personas.")

    # --- CASO 2: pregunta directa sobre un país ---
    m = re.search(r"(capital|poblaci[oó]n|superficie)\s+de\s+([a-záéíóúñ]+)", pregunta_low)
    if m:
        campo_raw, pais = m.group(1), m.group(2)
        campo = {"capital": "capital", "población": "poblacion",
                 "poblacion": "poblacion", "superficie": "superficie_km2"}[campo_raw]
        if n_obs == 0:
            return (f"Thought: Consulto directamente la base de conocimiento.\n"
                    f"Action: base_conocimiento[{pais}:{campo}]")
        else:
            resultado = observaciones[-1].replace("Observation:", "").strip()
            return f"Thought: Tengo la respuesta.\nFinal Answer: {resultado}"

    # --- CASO 3: cálculo puro ---
    m = re.search(r"cu[aá]nto es\s+(.+?)\??$", pregunta_low)
    if m and n_obs == 0:
        expr = m.group(1).replace("x", "*").replace("por", "*").replace("mas", "+")
        expr = re.sub(r"[^\d+\-*/.() ]", "", expr)
        return (f"Thought: Es un cálculo puro, uso la calculadora.\n"
                f"Action: calculadora[{expr.strip()}]")
    if m and n_obs >= 1:
        return f"Thought: Ya calculé.\nFinal Answer: {observaciones[-1].replace('Observation:', '').strip()}"

    # --- CASO 4: Conversión de moneda ---
    m = re.search(r"cu[aá]nto son\s+(\d+)\s+euros.*d[oó]lares", pregunta_low)
    if m:
        cantidad = m.group(1)
        if n_obs == 0:
            return (f"Thought: Me piden convertir {cantidad} euros a dólares.\n"
                    f"Action: conversor_moneda[{cantidad}]")
        else:
            resultado = observaciones[-1].replace("Observation:", "").strip()
            return f"Thought: Ya tengo la conversión.\nFinal Answer: {cantidad} euros son {resultado} dólares."

    # --- CASO 5: Densidad de población (3 herramientas) ---
    m = re.search(r"densidad.*poblaci[oó]n.*de\s+([a-záéíóúñ]+)", pregunta_low)
    if m:
        pais = m.group(1)

        if n_obs == 0:
            # Paso 1: Pedir población
            return (f"Thought: Para la densidad necesito la población de {pais}.\n"
                    f"Action: base_conocimiento[{pais}:poblacion]")

        elif n_obs == 1:
            # Paso 2: Pedir superficie
            return (f"Thought: Ya tengo la población. Ahora necesito la superficie de {pais}.\n"
                    f"Action: base_conocimiento[{pais}:superficie_km2]")

        elif n_obs == 2:
            # Paso 3: Calcular (Población / Superficie)
            poblacion = observaciones[0].replace("Observation:", "").strip()
            superficie = observaciones[1].replace("Observation:", "").strip()
            return (f"Thought: Tengo ambos datos ({poblacion} y {superficie}). Los divido.\n"
                    f"Action: calculadora[{poblacion} / {superficie}]")

        else:
            # Paso 4: Respuesta Final
            resultado = observaciones[-1].replace("Observation:", "").strip()
            return (f"Thought: Ya calculé la densidad.\n"
                    f"Final Answer: La densidad de población de {pais} es de {float(resultado):.2f} hab/km2.")


    # --- Fallback ---
    return ("Thought: No sé resolver esta pregunta con las herramientas disponibles.\n"
            "Final Answer: No puedo responder esa pregunta con las herramientas que tengo.")

print("✅ Cerebro por reglas definido.")

✅ Cerebro por reglas definido.


### 📝 Ejercicio 4 (reflexión) — ¿Dónde está el límite?

Responde en una celda markdown:
- ¿Por qué el cerebro basado en reglas **no escala**?
- ¿Qué problemas aparecen cuando las preguntas son abiertas o ambiguas?
- ¿Qué resuelve un LLM que las reglas no?

- ¿Por qué el cerebro basado en reglas no escala?

Porque el lenguaje natural es **infinito**. No puedes escribir una expresión regular (regex) para cada forma en la que un humano formula una pregunta.

**Ejemplo:**  
La pregunta "¿Y cuánta gente hay apretada en un kilómetro en España?" rompería cualquier regla de densidad preprogramada. El código acabaría siendo un bloque inmanejable de miles de `if/else`.

---

- ¿Qué problemas aparecen cuando las preguntas son abiertas o ambiguas?

El sistema se rompe o cae directamente en el **Fallback**.

Si la entrada tiene faltas de ortografía imprevistas, o requiere deducción en lugar de una extracción directa, las reglas fijas no saben cómo reaccionar.

---

- ¿Qué resuelve un LLM que las reglas no?

El LLM aporta **comprensión semántica** y **adaptabilidad**.

- Entiende el **significado** y la **intención** de la pregunta independientemente de cómo esté fraseada.
- Puede inferir qué herramienta usar simplemente leyendo su descripción, en lugar de necesitar un mapeo rígido programado de antemano.

## **8. (Opcional) Sustituir el cerebro por un LLM real**

Si quieres ir más allá, aquí tienes el esqueleto para usar un LLM real como cerebro. **Aviso:** `gpt2-spanish` no está entrenado para seguir el formato ReAct, así que los resultados serán imperfectos — ese es precisamente el punto didáctico: **los agentes de verdad usan modelos grandes entrenados con *instruction tuning* y *tool use*** (Claude, GPT-4, Llama-3-Instruct...).

Es muy útil ejecutar esto para ver **por qué** los modelos pequeños no sirven como cerebro agéntico.

In [13]:
from transformers import pipeline

# Inicializamos el LLM local
llm = pipeline("text-generation", model="datificate/gpt2-small-spanish", max_new_tokens=80)

PROMPT_TEMPLATE = """Eres un agente que responde preguntas usando herramientas.

Herramientas disponibles:
- calculadora[expresion]: evalúa una expresión matemática.
- base_conocimiento[pais:campo]: consulta datos de un país.

Formato obligatorio:
Thought: <tu razonamiento>
Action: <herramienta>[<input>]
O bien:
Thought: <tu razonamiento>
Final Answer: <respuesta>

Pregunta: {pregunta}
Historial:
{historial}

Siguiente paso:"""

def cerebro_llm(pregunta, historial):
    # Formateamos el prompt con la pregunta y el historial actual
    prompt = PROMPT_TEMPLATE.format(
        pregunta=pregunta,
        historial="\n".join(historial) if historial else "(vacío)"
    )

    # Le pedimos al LLM que genere la siguiente acción o pensamiento
    salida = llm(prompt, do_sample=True, temperature=0.3)[0]["generated_text"]

    # Extraemos solo la parte nueva generada por el modelo (quitamos el prompt original)
    return salida[len(prompt):].strip()

# Probamos el agente inyectándole nuestro nuevo "cerebro" basado en IA
agente("¿Cuál es la capital de España?", cerebro_fn=cerebro_llm)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/510M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🤔 PREGUNTA: ¿Cuál es la capital de España?

--- Paso 1 ---
Thought: <tu razonamiento>

Final Answer: <respuesta>

Pregunta: ¿Cuál es la capital de España?
Historial:
(vacío)

Siguiente paso:

Thought: <tu razonamiento>

Final Answer: <respuesta>

Pregunta: ¿Cuál

🎯 RESPUESTA: <respuesta>

Pregunta: ¿Cuál


'<respuesta>\n\nPregunta: ¿Cuál'

## **9. Conclusiones**

Lo que has construido en este notebook es, estructuralmente, **un agente**. No es un juguete: es exactamente el mismo patrón que usan sistemas en producción.

**Ideas clave:**

- Un **agente** = un **bucle** que combina razonamiento, acción y observación.
- Las **herramientas** son simplemente funciones de Python. Cualquier API, base de datos o script puede convertirse en una herramienta.
- El **cerebro** puede ser cualquier cosa que sepa decidir el siguiente paso — desde reglas hasta un LLM grande. Cuanto mejor el cerebro, más complejas son las tareas que puede resolver.
- El patrón **ReAct** (Thought → Action → Observation) es la base de frameworks como LangChain, AutoGen, CrewAI o el propio "tool use" de Claude y GPT.

**Próximos pasos en el mundo real:**

| Limitación actual | Solución en agentes reales |
|---|---|
| Cerebro por reglas | LLM grande con *tool use* nativo (Claude, GPT-4, Llama-3) |
| Herramientas en memoria | APIs reales, bases de datos, web search, RAG |
| Una sola tarea | Memoria conversacional entre turnos |
| Un solo agente | Multi-agente (varios agentes colaborando) |
| Sin control de errores | Reintentos, validación, *guardrails* |

Has pasado de *generar texto* (Actividades 1 y 2) a **construir un sistema que actúa**. Esa es, en esencia, la diferencia entre un modelo y un agente.